## Proyecto final: Clasificación de señales ECG 
## Proyecto 2026-02 


## Presentado por:

 Esteban Garcia Otalvaro
 
 Fabian Andres Posso

Bioseñales y Sistemas — Bioingeniería, Universidad de Antioquia — 2026-02

Base de datos: Chapman University & Shaoxing People's Hospital (Zheng et al., 2020), disponible en Figshare: https://figshare.com/collections/ChapmanECG/4560497/1

In [12]:
# Importa el módulo os para interactuar con el sistema operativo y manejar rutas de archivos
import os

# Importa seaborn (como sns) para generar gráficas estadísticas y visualizaciones más atractivas
import seaborn as sns

# Importa numpy (como np) para realizar cálculos numéricos y manejar arreglos matemáticos
import numpy as np

# Importa pandas (como pd) para cargar, leer y manipular el archivo Excel como una tabla de datos (DataFrame)
import pandas as pd

# Importa el submódulo pyplot de matplotlib para generar gráficas visuales
import matplotlib.pyplot as plt

# Importa FancyBboxPatch para poder dibujar cajas con bordes redondeados dentro de las gráficas
from matplotlib.patches import FancyBboxPatch

# Importa las pruebas estadísticas de Chi-cuadrado y Fisher para analizar cruces de variables categóricas (ej. Género vs Arritmia)
from scipy.stats import chi2_contingency, fisher_exact

# Importa la prueba de Kruskal-Wallis (para comparar medianas) y skew (para calcular la asimetría de los datos)
from scipy.stats import kruskal, skew

# Importa la distribución Chi-cuadrado y la renombra como dist_chi2 para que no haga conflicto con otros nombres
from scipy.stats import chi2 as dist_chi2

# Importa la función iqr (rango intercuartílico) de scipy.stats para calcular la dispersión de los datos
from scipy.stats import kruskal, chi2_contingency, iqr

import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Rutas y cargue de archivos

# 1. Definimos la ruta de la carpeta base
CARPETA_BASE = r"C:\Users\12345\Downloads\UDEA\BIOSEÑALES\PROYECTOS\ECG"

# 2. Archivo con etiquetas y datos clínicos 
RUTA_DIAGNOSTICOS = os.path.join(CARPETA_BASE, "Diagnostics.xlsx")

# 3. Carpeta con los .csv de las señales 
CARPETA_ECG = r"C:\Users\12345\Downloads\UDEA\BIOSEÑALES\PROYECTOS\ECG\ECGData"

# Define la frecuencia de muestreo constante de la base de datos (500 Hz = 500 datos por segundo)
FS = 500

# Ajusta la configuración de las gráficas para que tengan mayor nitidez (110 DPI) y un tamaño de letra base de 10
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# Imprime la ruta de la carpeta actual donde tienes abierto este Jupyter Notebook
print("Carpeta del notebook:       ", os.getcwd())

# Verifica si el archivo de Excel existe en la ruta que armamos arriba (debe imprimir True)
print("Diagnostics.xlsx encontrado:", os.path.exists(RUTA_DIAGNOSTICOS))

# Verifica si la subcarpeta con los ECG existe en la ruta que armamos arriba (debe imprimir True)
print("Carpeta ECG encontrada:     ", os.path.isdir(CARPETA_ECG))

Carpeta del notebook:        c:\Users\12345\OneDrive\Documentos\GitHub\Biose-ales-Esteban-Garcia\Proyecto Semestre II-Entrega Final
Diagnostics.xlsx encontrado: True
Carpeta ECG encontrada:      True


## 1.Revisión teórica del artículo "optimal Multi-Stage Arrhythmia Classification Approach"




Enlace de acceso artículo 1: https://www.nature.com/articles/s41598-020-59821-7

**Resumen de la sección "Extracción de caracteristicas:**

La idea central es que, en lugar de buscar y medir cada onda del latido (P, Q, R, S, T), los autores miden todos los picos y valles de la señal sin preocuparse por qué onda es cada uno, y resumen esas medidas en histogramas. Así el método sigue funcionando aunque falte alguna onda.

A continuación se explican las razones de no usar las herramientas alternativas:

• Medir cada onda por separado (su altura, su duración, la distancia entre ellas) falla en las arritmias donde esas ondas se deforman o desaparecen.

• Redes neuronales que aprenden las características solas necesitan muchos datos y mucho cómputo, y no se puede saber qué está midiendo la red.

• Fourier o wavelets indican qué frecuencias hay, pero no cuándo ocurren.

A continuación se explica el paso a paso del método propuesto, que es interpretable y se aplica a la derivación II:

**1.  Reescalado.** Cada señal se normaliza por máximo–mínimo para que su pico máximo sea 1. Con esto se compensan las diferencias de amplitud entre personas (sexo, edad, raza). El efecto sobre la clasificación es pequeño pero positivo: el F1 sube entre 0,001 y 0,0016.

**2.  Picos y valles.** Se toman todos los máximos y mínimos locales de la señal filtrada. De cada uno se miden la altura, el ancho y la prominencia, que es la distancia vertical entre el pico y su línea de contorno más baja.

**3.  Tres subconjuntos.** Cada pico o valle se asigna a uno de tres grupos: complejos QRS, picos que no son QRS y valles.

**4.  Seis combinaciones por pares.** Se comparan los elementos de a dos: QRS–QRS, no-QRS–no-QRS, valle–valle, QRS–no-QRS, QRS–valle y no-QRS–valle. Para cada par se calcula la diferencia de ubicación en el tiempo y tres cocientes: diferencia de ancho, de altura y de prominencia, cada una dividida por la diferencia de tiempo.

**5.  Vector de longitud fija.** Cada paciente tiene un número distinto de picos y de cocientes, así que no se pueden usar directamente. Por eso se construye una tabla de frecuencias empírica con 100 clases entre el mínimo y el máximo de cada variable (Figs. 15–17 del artículo).

**6.  Características adicionales.** Se agregan la edad, el sexo, la media y la varianza de los intervalos RR, el número de intervalos RR y las mediciones del equipo: frecuencia ventricular y auricular, duración del QRS, intervalo QT, eje R, eje T, conteo de QRS y los inicios y finales de la onda Q.

**Resultado.** Con las mismas condiciones, los modelos con las características diseñadas en la derivación II (Grupo 2) obtienen un F1 promedio 0,021 mayor que los que usan solo las mediciones del equipo (Grupo 1).

**Cómo se arma el vector de características**

El Grupo 6 (solo derivación II) es el que concentra las 3.302 variables del artículo. Cada bloque son histogramas de 100 clases:

In [13]:
N_CLASES = 100                      # clases de cada histograma
N_ATRIBUTOS = 3                     # altura, ancho y prominencia
N_SUBCONJUNTOS = 3                  # QRS, picos no-QRS y valles
N_PARES = 6                         # QRS-QRS, noQRS-noQRS, valle-valle, QRS-noQRS, QRS-valle, noQRS-valle
N_COCIENTES = 3                     # altura/tiempo, ancho/tiempo, prominencia/tiempo

tabla_grupo6 = pd.DataFrame([
    ['Altura, ancho y prominencia de QRS, picos no-QRS y valles',
     '3 atributos × 3 subconjuntos × 100 clases', N_ATRIBUTOS * N_SUBCONJUNTOS * N_CLASES],
    ['Diferencia de ubicación entre pares',
     '6 combinaciones × 100 clases', N_PARES * N_CLASES],
    ['Cocientes (altura, ancho y prominencia sobre diferencia de tiempo)',
     '3 cocientes × 6 combinaciones × 100 clases', N_COCIENTES * N_PARES * N_CLASES],
    ['Edad y sexo (deducido: 3.302 − 3.300)', '2 variables', 2],
], columns=['Bloque', 'Cuenta', 'Variables'])

total = tabla_grupo6['Variables'].sum()
tabla_grupo6.loc[len(tabla_grupo6)] = ['Total', '', total]
assert total == 3302, "El total debe coincidir con el Grupo 6 del artículo (3.302)"
tabla_grupo6

def centrar(df):
    return (df.style
              .set_properties(**{'text-align': 'center'})
              .set_table_styles([{'selector': 'th', 'props': [('text-align', 'center')]}])
              .hide(axis='index'))
    
display(centrar(tabla_grupo6))

Bloque,Cuenta,Variables
"Altura, ancho y prominencia de QRS, picos no-QRS y valles",3 atributos × 3 subconjuntos × 100 clases,900
Diferencia de ubicación entre pares,6 combinaciones × 100 clases,600
"Cocientes (altura, ancho y prominencia sobre diferencia de tiempo)",3 cocientes × 6 combinaciones × 100 clases,1800
Edad y sexo (deducido: 3.302 − 3.300),2 variables,2
Total,,3302


La siguiente tabla incluye todos los grupos de caracteristicas del artículo (cada grupo que se pudo formar es una mezcla de los bloques de medición del equipo, estadisticas simples e histogramas de la primera tabla):

In [14]:
tabla_grupos = pd.DataFrame({
    'Grupo': range(1, 12),
    'Contenido': [
        'Mediciones del equipo (frecuencia ventricular y auricular, duración QRS, QT, ejes R y T, conteo QRS, inicio y fin de Q)',
        'Media y varianza de RR, nº de intervalos RR, media y varianza de altura, ancho y prominencia de QRS, picos no-QRS y valles (derivación II)',
        'Grupo 1 + Grupo 2',
        'Grupo 2 ampliado a las 12 derivaciones',
        'Grupo 1 + Grupo 4',
        'Histogramas de la derivación II (Tabla 1)',
        'Grupo 2 + Grupo 6',
        'Grupos 1 + 2 + 6',
        'Grupo 6 en las 12 derivaciones',
        'Grupo 4 + Grupo 9',
        'Grupo 3 + Grupo 9',
    ],
    'Origen de los datos': [
        'Medición del equipo', 'Derivación II', 'Equipo + derivación II', 'Las 12 derivaciones',
        'Equipo + 12 derivaciones', 'Derivación II', 'Derivación II', 'Equipo + derivación II',
        'Las 12 derivaciones', 'Las 12 derivaciones', 'Equipo + 12 derivaciones'],
    'Variables': [11, 23, 32, 221, 230, 3302, 3323, 3332, 39602, 39821, 39830],
})
tabla_grupos

display(centrar(tabla_grupos))

Grupo,Contenido,Origen de los datos,Variables
1,"Mediciones del equipo (frecuencia ventricular y auricular, duración QRS, QT, ejes R y T, conteo QRS, inicio y fin de Q)",Medición del equipo,11
2,"Media y varianza de RR, nº de intervalos RR, media y varianza de altura, ancho y prominencia de QRS, picos no-QRS y valles (derivación II)",Derivación II,23
3,Grupo 1 + Grupo 2,Equipo + derivación II,32
4,Grupo 2 ampliado a las 12 derivaciones,Las 12 derivaciones,221
5,Grupo 1 + Grupo 4,Equipo + 12 derivaciones,230
6,Histogramas de la derivación II (Tabla 1),Derivación II,3302
7,Grupo 2 + Grupo 6,Derivación II,3323
8,Grupos 1 + 2 + 6,Equipo + derivación II,3332
9,Grupo 6 en las 12 derivaciones,Las 12 derivaciones,39602
10,Grupo 4 + Grupo 9,Las 12 derivaciones,39821


**¿Qué se hace cuando desaparece una onda del complejo PQRST?**

El caso que cita el articulo es la fibrilación auricular (AFIB) y el aleteo auricular (AFL): sus ondas P se reemplazan por múltiples ondas de aleteo o fibrilación, más bajas que una P normal y sin relación con el ritmo de los QRS. Las características clásicas, que miden P, Q y T como ondas con nombre, pierden información justo en estas arritmias "caracterizadas por distorsión o por omisión completa de algunos componentes". Su respuesta es no depender de reconocer cada onda: los picos y valles son máximos y mínimos locales, y solo se clasifican en QRS, picos no-QRS y valles.
Cómo lo resuelve el diseño:

• Una onda ausente no deja un hueco; si no hay onda P, no hay un valor faltante que imputar; la distribución de picos no-QRS cambia de forma, y el vector de 100 clases sigue siendo válido.

• Las ondas f (múltiples ondas de aleteo y de fibrilación) de la AFIB se miden como lo que son; aparecen como muchos picos no-QRS de baja prominencia, sin distancia fija al QRS, eso ensancha los histogramas de diferencia de ubicación y de prominencia entre QRS y no-QRS, frente a un ritmo sinusal donde esa distancia se concentra en unas pocas clases.

• La irregularidad del ritmo se mide aparte, con la media, la varianza y el conteo de los intervalos RR. Esas tres cantidades solo necesitan localizar los QRS, así que no se ven afectadas por la ausencia de P.

• El histograma tiene longitud fija. No importa cuántas ondas tenga cada latido o paciente.
Límites que el artículo reconoce. Las alteraciones de conducción que cambian la forma del QRS (extrasístoles ventriculares, bloqueos de rama derecha e izquierda, extrasístoles auriculares) bajan el F1 promedio entre 0,017 y 0,034, así que el método no es inmune a ellas.

## 3.Flujo de filtrado y comparación de señales entre ambos proyectos:

La base de datos ChapmanECG es un repositorio clínico a gran escala que contiene electrocardiogramas de 12 derivaciones de 10,646 pacientes, registrados en ventanas temporales de 10 segundos a una frecuencia de muestreo de 500 Hz. Todos los datos fueron evaluados y etiquetados rigurosamente por cardiólogos profesionales para clasificar 11 ritmos cardíacos principales y más de 60 condiciones cardiovasculares adicionales, convirtiéndola en un recurso estandarizado de alta fidelidad, ideal para entrenar y validar modelos de inteligencia artificial orientados al diagnóstico automatizado de arritmias. [4]

El archivo Diagnostics.xlsx estructura sus 16 columnas en tres grupos esenciales: la identificación y demografía del paciente (FileName, PatientAge, Gender), las etiquetas diagnósticas establecidas por cardiólogos (Rhythm, Beat), y los parámetros fisiológicos detallados extraídos del electrocardiograma, como frecuencias cardíacas, duraciones de intervalos (QRS, QT) y ejes eléctricos. Esta tabla funciona como la base de referencia (ground truth) que vincula de manera estructurada cada señal con su respectiva condición médica y métricas clínicas.

## ¿Qué tipo de arritmias son más prevalentes? 

Según los resultados obtenidos, excluyendo el Ritmo Sinusal normal (SR) que representa a los pacientes sanos (1,826 casos), las arritmias más prevalentes en la base de datos son la Bradicardia Sinusal (SB) con 3,889 casos, seguida de la Fibrilación Auricular (AFIB) con 1,780 casos y la Taquicardia Sinusal (ST) con 1,568 casos. Estas tres anomalías concentran la inmensa mayoría de las patologías de la muestra, dejando al resto de irregularidades cardíacas con una representación estadísticamente minoritaria.

## ¿Hay arritmias que son más prevalentes para un género específico?

Debido al desbalance poblacional en la base de datos (mayor cantidad de hombres), la prevalencia de arritmias debe compararse utilizando proporciones relativas dentro de cada sexo en lugar de conteos absolutos. Para garantizar que las diferencias observadas sean estadísticamente significativas y no producto del azar, la metodología emplea dos niveles de validación: primero, una prueba global de Chi-cuadrado (excluyendo clases minoritarias para cumplir los supuestos matemáticos y midiendo el efecto con la V de Cramér) y segundo, pruebas individuales para cada ritmo (Chi-cuadrado o Fisher), aplicando la estricta corrección de Bonferroni a los valores p para mitigar el riesgo de falsos positivos al realizar múltiples comparaciones simultáneas.

Se formulas las siguientes hipóteesis:

-Hipótesis nula ($H_0$): La distribución de los ritmos cardíacos es la misma en ambos sexos, lo que significa que el tipo de arritmia es estadísticamente independiente del género del paciente.

-Hipótesis alternativa:($H_1$): La distribución de los ritmos cardíacos no es la misma en ambos sexos, lo que indica que existe una dependencia estadística significativa entre el género y el tipo de arritmia desarrollado.

## Interpretación de los resultados

El valor p obtenido (9.38e-49) es extremadamente bajo, lo que permite rechazar la hipótesis nula y confirmar contundentemente que existe una relación estadísticamente significativa entre el género y el tipo de arritmia. Sin embargo, el valor de la V de Cramér (0.151) indica que el tamaño de este efecto es pequeño, lo que significa que la asociación, aunque matemáticamente real, es débil y el género no es un factor determinante por sí solo. 

## Taquicardias y predisposición femenina (ST, SVT)
La mayor prevalencia de Taquicardia Sinusal (ST) y Taquicardia Supraventricular (SVT) en mujeres concuerda sólidamente con la literatura médica. Estudios electrofisiológicos demuestran que las mujeres presentan, en promedio, una frecuencia cardíaca basal más alta (entre 3 y 5 latidos por minuto mayor que los hombres) y un período refractario del nodo auriculoventricular más corto [5]. Estas diferencias, mediadas por los efectos de los estrógenos sobre los canales iónicos, incrementan de manera natural la susceptibilidad del corazón femenino a desarrollar taquicardias, especialmente aquellas causadas por circuitos de reentrada [6].

## Bradicardia y predisposición masculina (SB)
La mayor frecuencia de Bradicardia Sinusal (SB) en la cohorte masculina se alinea con investigaciones poblacionales que documentan un mayor tono vagal (parasimpático) intrínseco y una menor tasa de despolarización espontánea del nodo sinusal en hombres [1], [2]. Esta condición suele ser fisiológica y se asocia frecuentemente con diferencias en la masa muscular y el acondicionamiento cardiovascular.

## Limitación fisiológica y demográfica (Factor de confusión por edad)
El análisis actual asume que las diferencias en las arritmias se deben puramente al sexo biológico, omitiendo el control de la edad como variable clínica fundamental. Fisiológicamente, el envejecimiento provoca fibrosis miocárdica, degeneración del sistema de conducción y alteraciones estructurales que disparan la incidencia de arritmias, especialmente la Fibrilación Auricular (AFIB). En la bases datos el promedio de edad de la mujeres es de 58,05 años y el de los hombres de 60,08 años.

## ¿Hay arritmias que son más prevalentes para una edad?

Tras confirmar matemáticamente mediante el cálculo de asimetría (-0.61) que la mayoría de los pacientes se concentran en edades avanzadas, la variable continua de la edad se transformó en rangos categóricos ordinales utilizando límites predefinidos (menores de 18, 18 a 30, y décadas sucesivas hasta mayores de 80). Mediante la función pd.cut, cada paciente fue asignado a su respectivo intervalo temporal, lo que permitió agrupar la base de datos para extraer las frecuencias absolutas (n) y relativas (%) de cada grupo, consolidando así toda la distribución demográfica en una única tabla descriptiva.

## Interpretacion

**Significancia y Validación Estadística:** El valor p obtenido en la prueba de Chi-cuadrado (< 1e-300) indica que existe una dependencia estadísticamente significativa entre el grupo de edad y el tipo de ritmo desarrollado. Desde el punto de vista metodológico, la prueba es completamente válida: solo el 1.6% de las celdas (1 de 64) presentó una frecuencia esperada menor a 5, y el mínimo esperado fue de 3.01. Esto cumple estrictamente con el criterio estadístico de aceptar hasta un 20% de celdas con valores esperados menores a 5 [7], garantizando que el tamaño de la muestra es robusto.

**Patrones Clínicos en Población Joven (<30 años):** Los datos demuestran que las alteraciones del ritmo en pacientes jóvenes están ligadas principalmente al automatismo sinusal. En el grupo pediátrico y adolescente (<18 años), predomina fuertemente la Arritmia Sinusal (SA) con un 45.7%, lo cual concuerda con la literatura médica que cataloga este ritmo como un hallazgo fisiológico normal y benigno en jóvenes, asociado a la variabilidad de la frecuencia cardíaca inducida por la respiración [8]. Entre los 18 y 30 años, los ritmos predominantes son la Taquicardia Sinusal (ST) con 26.8% y la Bradicardia Sinusal (SB) con 24.7%.

**Transición Patológica en Edad Avanzada (>60 años):** El mapa de calor revela una clara transición epidemiológica impulsada por el envejecimiento cardiovascular. A partir de los 60 años, la incidencia de arritmias estructurales se dispara debido a la fibrosis miocárdica celular y la degeneración del sistema de conducción [9]. La Fibrilación Auricular (AFIB), que tiene una presencia casi nula en pacientes jóvenes, crece exponencialmente década tras década hasta convertirse en la arritmia dominante absoluta [7], [3]. En los pacientes mayores de 80 años, el 44.8% de los diagnósticos corresponden a AFIB, desplazando por completo al Ritmo Sinusal (SR), que cae a apenas un 7.7%.

**Magnitud de la Asociación (V de Cramér):** El valor de la V de Cramér (0.231) clasifica el tamaño del efecto en el umbral entre pequeño y moderado. Sin embargo, al comparar este resultado con el obtenido previamente en la variable de género (V = 0.151), queda en evidencia que la edad tiene un peso predictivo mucho mayor que el sexo biológico para determinar el perfil arrítmico de un paciente [7]. El envejecimiento actúa como el principal catalizador en esta cohorte para la aparición de arritmias complejas.

**Contraste con la evidencia científica:** El incremento progresivo de la fibrilación auricular (FA) en pacientes mayores respalda los hallazgos de investigaciones clínicas previas, como el estudio ATRIA. Sin embargo, la incidencia reportada en esta base de datos es significativamente superior a la de la población general debido a un sesgo de selección: la muestra está compuesta exclusivamente por pacientes hospitalarios con una indicación médica previa para realizarse un electrocardiograma. Por otro lado, la notable concentración de arritmias sinusales en la población joven representa un comportamiento fisiológico normal, ya que la irregularidad del ritmo impulsada por el ciclo respiratorio es naturalmente más intensa en niños y adultos jóvenes.   

**Sesgo diagnóstico en pacientes pediátricos (<18 años):** Es fundamental reconocer que los rangos normales de frecuencia cardíaca en niños son inherentemente más elevados que en los adultos. Esto indica que el 32,1 % de los diagnósticos de "taquicardia sinusal" en pacientes menores de 18 años es, muy probablemente, un error de clasificación causado por evaluar ritmos pediátricos saludables utilizando umbrales diagnósticos de adultos (frecuencias superiores a 100 lpm). Para evitar que este sesgo contamine los patrones de los algoritmos de aprendizaje, se recomienda metodológicamente excluir a la población menor de 18 años del conjunto de datos definitivo. 

## Los algoritmos de aprendizaje se basan en usar la mayor cantidad de datos posibles que sean comparables, ¿Qué rango de edades y arritmias de la base de datos entregada considera que debe usar para entrenar algoritmos de aprendizaje?

Para entrenar un modelo de inteligencia artificial capaz de procesar señales ECG y generar recomendaciones clínicas precisas, la selección del conjunto de datos debe aislar la variabilidad fisiológica extrema y el sesgo estadístico.

Rango de edades recomendado: Adultos (≥ 18 años)

**Exclusión de población pediátrica y juvenil(<18 años):** Este grupo representa apenas el 2.49% de los datos y presenta un perfil eléctrico basal distinto, dominado por la Arritmia Sinusal (SA) como hallazgo fisiológico. Incluirlos forzaría al algoritmo a intentar generalizar excepciones con muy pocos ejemplos.

**Núcleo de entrenamiento:** Focalizar el modelo en pacientes mayores de 18 años (o incluso mayores de 30 años, donde comienza el mayor volumen de datos) concentra el aprendizaje en la matriz de datos más densa, garantizando que el algoritmo asimile patrones morfológicos consistentes de arritmias estructurales y de conducción típicas del adulto.

**Arritmias recomendadas: 8 clases principales**

**Exclusión de clases ultra-minoritarias (AVNRT, AVRT, SAAWR):** Con 16, 8 y 7 casos respectivamente, carecen del volumen mínimo viable para el aprendizaje profundo. Forzar su inclusión provocará inevitablemente un sobreajuste (overfitting) severo y confusión en la matriz de confusión del modelo.

**Inclusión de las clases base:** Conservar Bradicardia Sinusal (SB), Ritmo Sinusal (SR), Fibrilación Auricular (AFIB), Taquicardia Sinusal (ST), Taquicardia Supraventricular (SVT), Aleteo Auricular (AF), Arritmia Sinusal (SA) y Taquicardia Auricular (AT).

**Gestión del desbalance:** Dado que la cohorte recomendada presenta una asimetría extrema (ej. 3889 casos de SB frente a 121 casos de AT), será obligatorio implementar técnicas en el preprocesamiento —como sobremuestreo de clases minoritarias, submuestreo cuidadoso de SB, o funciones de pérdida ponderada (weighted loss)— para evitar que el algoritmo clasifique todo como Bradicardia Sinusal por simple probabilidad estadística.

Para asegurar el entrenamiento óptimo del algoritmo con datos comparables, el subconjunto metodológico excluye a los menores de 18 años, evitando así que sus altas frecuencias cardíacas normales introduzcan falsos diagnósticos de taquicardia basados en umbrales de adulto, sin fijar un límite superior para preservar los vitales casos de fibrilación auricular en la tercera edad. Simultáneamente, para solucionar la escasez de ejemplos, las arritmias se reestructuran agrupando clases minoritarias o clínicamente equivalentes en categorías robustas consolidadas (SB, AFIB, GSVT, SR y ST). Finalmente, se decidió conservar los registros de pacientes con condiciones cardíacas adicionales para no reducir drásticamente la muestra disponible (especialmente en los casos de fibrilación auricular), delegando el control de este factor a futuros análisis de sensibilidad.

El subconjunto final de datos destinado al entrenamiento de los algoritmos se restringe a pacientes de 18 años o más y consolida las señales en cinco categorías clínicas principales para garantizar un volumen robusto de ejemplos y evitar la confusión algorítmica entre diagnósticos equivalentes [2] [4]:

SB (Bradicardia Sinusal): Constituye el grupo mayoritario (37.29 %, n=3871), compuesto exclusivamente por casos de bradicardia pura con una mediana de edad de 60 años.

AFIB (Fibrilación y Flutter Auricular): Representa el 21.43 % (n=2225) y concentra a la población más envejecida (mediana de 74 años). Unifica ambas patologías debido a su frecuente coexistencia clínica, tratamiento compartido y alta probabilidad de solapamiento en modelos computacionales.

SR (Ritmo Sinusal): Establece la línea base de normalidad al agrupar el ritmo sinusal estándar y la arritmia/irregularidad sinusal (20.05 %, n=2081).

ST (Taquicardia Sinusal): Se conserva de forma provisional como una categoría independiente (14.29 %, n=1483), cuyo uso definitivo está sujeto a validación mediante el parámetro de control ST_EN_GSVT.

GSVT (Taquicardias Supraventriculares Globales): Es el grupo más compacto (6.95 %, n=721) y soluciona la falta de datos de las clases ultra-minoritarias al integrar en una sola etiqueta la taquicardia supraventricular (SVT), taquicardia auricular (AT), AVNRT, AVRT y el marcapasos auricular migratorio (SAAWR).

¿Hay coincidencia entre lo encontrado en literatura y lo encontrado en la base de datos? 

El análisis comparativo revela que ambas bases de datos son altamente consistentes en las arritmias más prevalentes, presentando variaciones menores al 2 % en su frecuencia y diferencias de edad promedio generalmente inferiores a un año. La discrepancia epidemiológica más destacada es una sobre-representación del 4.17 % en los casos de Taquicardia Supraventricular (SVT) dentro de la base actual. En cuanto a la distribución por sexo, mientras los ritmos principales (SB, AFIB, ST) mantienen diferencias menores al 1 %, las clases ultra-minoritarias (como AVNRT y SAAWR) muestran desviaciones extremas de hasta -71.43 % en la proporción de hombres; sin embargo, esto no representa una divergencia clínica real, sino un artefacto estadístico derivado del tamaño de muestra críticamente bajo (n < 20) en estas categorías específicas.

In [38]:
# 1. CREAR EL DATAFRAME FALTANTE
# Recreamos la Tabla 6 ingresando los totales y los hombres por ritmo
datos_tabla_6 = {
    'Rhythm': ['SB', 'SR', 'AFIB', 'ST', 'AF', 'SA', 'SVT', 'AT', 'AVNRT', 'AVRT', 'SAAWR'],
    'Total': [15528, 7291, 7029, 6208, 1723, 1771, 543, 133, 16, 8, 8],
    'Hombres': [9845, 4107, 4051, 3208, 1000, 978, 290, 69, 12, 5, 6]
}
df_tabla_6 = pd.DataFrame(datos_tabla_6)

# 2. AUDITORÍA DE CONSISTENCIA
total_hombres_texto = 22599
suma_hombres_tabla = df_tabla_6['Hombres'].sum()
diferencia = suma_hombres_tabla - total_hombres_texto

print(f"Total de hombres reportado en el texto de [2]: {total_hombres_texto}")
print(f"Suma de la columna de hombres de la Tabla 6:   {suma_hombres_tabla}")
print(f"Diferencia:                                    {diferencia}\n")

# 3. CORREGIR EL ERROR DEL ARTÍCULO (Intercambiar Hombres/Mujeres)
ritmos_invertidos = ['SR', 'SVT', 'AVNRT', 'SAAWR']

# Calculamos las mujeres reales (Total - Hombres) y las reasignamos como Hombres
mascara = df_tabla_6['Rhythm'].isin(ritmos_invertidos)
df_tabla_6.loc[mascara, 'Hombres'] = df_tabla_6.loc[mascara, 'Total'] - df_tabla_6.loc[mascara, 'Hombres']

# 4. RECALCULAR LA SUMA AJUSTADA
suma_corregida = df_tabla_6['Hombres'].sum()

print(f"Suma tras intercambiar hombres/mujeres en {ritmos_invertidos}: {suma_corregida}")

Total de hombres reportado en el texto de [2]: 22599
Suma de la columna de hombres de la Tabla 6:   23571
Diferencia:                                    972

Suma tras intercambiar hombres/mujeres en ['SR', 'SVT', 'AVNRT', 'SAAWR']: 22599


Análisis y Validación de ResultadosRepresentatividad confirmada de la muestra:

 -Los datos demográficos de nuestra base concuerdan fielmente con los reportes de los autores originales (Zheng et al. [4]), replicando con exactitud la cantidad de pacientes por sexo y los porcentajes de edad. Al comparar nuestra muestra con el macro-conjunto de 40.258 registros [2], el orden de prevalencia de las arritmias principales (SB > SR > AFIB > ST) se mantiene idéntico, y las medias de edad muestran variaciones ínfimas (inferiores a un año, salvo en la arritmia sinusal), lo que ratifica que esta base pública es una muestra estadísticamente representativa.   
 
 -Corrección del sesgo tipográfico en la literatura: Se evidenció una inconsistencia matemática en la Tabla 6 de la publicación de referencia [2], ya que la suma de pacientes masculinos en dicha tabla sobrepasaba el total declarado por los mismos autores en el texto. Al invertir computacionalmente las columnas de hombres y mujeres para los ritmos SR, SVT, AVNRT y WAP, las cifras cuadran a la perfección y se alinean con nuestra base de datos, lo que demuestra que el error provino de la publicación original y ratifica que el ritmo sinusal y la SVT son efectivamente más comunes en mujeres.   
 
 -Discrepancia menor en los registros de SVT: Se detectó que nuestra base pública contiene ligeramente más casos de SVT (587 casos) y AVRT que el conjunto total de 40.258 pacientes. Esto sugiere que hubo un proceso de reclasificación o corrección de etiquetas médicas entre una publicación y otra. No obstante, este detalle es intrascendente para el entrenamiento del modelo, dado que todas estas subclases serán absorbidas por el macrogrupo GSVT.   
 
 -Consistencia Epidemiológica y Sesgo de Selección: Si bien las trayectorias de las arritmias en relación con la edad y el sexo encajan con la literatura médica, las tasas absolutas de prevalencia están sobredimensionadas en nuestros datos (ej. 16,7 % de Fibrilación Auricular en la base frente a un 0,95 % estimado en la población general). Esto responde a un sesgo clínico inherente: los datos no reflejan a personas sanas, sino a pacientes de un entorno hospitalario con indicaciones médicas previas para un electrocardiograma. Por consiguiente, estas cifras son útiles para machine learning pero no para estimaciones demográficas globales.

 | Hallazgo en esta base [4] | Literatura | ¿Coincide con [2] y literatura? |
| :--- | :--- | :--- |
| SB más frecuente en hombres; ST y SR más frecuentes en mujeres | Las mujeres tienen una frecuencia cardíaca en reposo de 3 a 5 lpm mayor que los hombres [2] | Sí |
| SVT más frecuente en mujeres | La taquicardia por reentrada nodal es cerca de 2 veces más frecuente en mujeres [2] | Sí |
| AFIB algo más frecuente en hombres (tendencia, no significativa tras Bonferroni) | La FA es cerca de 1,5 veces más común en hombres [1] | Parcialmente |
| AFIB aumenta con la edad (0 % en < 18 años, 44,8 % en > 80 años) | La prevalencia de FA pasa de 0,1 % en < 55 años a 9,0 % en ≥ 80 años [3] | Sí (tendencia) |
| SA es la arritmia de menor edad (mediana 29 años) | La arritmia sinusal respiratoria es más marcada en niños y adultos jóvenes [4] | Sí |
| 32,1 % de "taquicardia sinusal" en < 18 años | La frecuencia cardíaca normal en niños es mayor que en adultos [4] | Sí (explica el hallazgo) |

## ¿Qué otros análisis propone desde la base de datos?

Los parámetros de control y las variables preservadas durante la depuración de la base de datos establecen las siguientes líneas de análisis posteriores para optimizar el entrenamiento algorítmico y su aplicación clínica:

-Análisis de sensibilidad por comorbilidades: Mediante la activación del parámetro SOLO_SIN_CONDICIONES, se entrenarán modelos paralelos para aislar el impacto de las patologías secundarias (columna Beat). Esto permitirá cuantificar si la mejora en la exactitud del algoritmo al usar señales puras compensa la eliminación de más de la mitad de los registros, especialmente en la Fibrilación Auricular (AFIB).  

-Validación de clasificación para la Taquicardia Sinusal (ST): Utilizando el interruptor lógico ST_EN_GSVT, se realizarán pruebas de rendimiento para determinar empíricamente si mantener la ST como un quinto grupo independiente aporta valor predictivo, o si genera excesiva confusión algorítmica y debe fusionarse definitivamente dentro del macro-grupo GSVT.   

-Correlación hemodinámica mediante modelos Windkessel: Al contar con una matriz de ritmos consolidada, las clasificaciones eléctricas pueden integrarse en modelos matemáticos de dinámica de fluidos cardiovascular (como Windkessel de 3 y 4 elementos) para realizar la identificación de parámetros y evaluar el impacto mecánico de estas arritmias sobre la perfusión y la presión aórtica.Integración en motores de asistencia clínica (IA): El subconjunto de datos final está estructurado para alimentar un pipeline integral de procesamiento de señales en Python. Esto facilita el desarrollo de asistentes de salud basados en inteligencia artificial que no solo clasifiquen el ECG multiderivación, sino que traduzcan automáticamente estos hallazgos en resúmenes clínicos y recomendaciones procesables.

Si se observa esta tabla y notas que los hombres siguen siendo ~63% en la SB sin importar el rango de edad, confirmas que la edad no está sesgando ese hallazgo.

Con base en el análisis descriptivo del punto anterior, seleccionar un subconjunto de datos para trabajar en los siguientes análisis. 
3. Programar las fórmulas 1 al 4 que se encuentran en el artículo (15%). 

Consultar que significado y utilidad del biespectro (5%) https://arxiv.org/pdf/1809.08451 

## 3. Fórmulas del artículo y significado-utilidad del biespectro

3. Programar las fórmulas 1 al 4 que se encuentran en el artículo (15%). 
Consultar que significado y utilidad del biespectro (5%)

In [40]:
# FÓRMULA 1: Desviación del ajuste de tendencia (Detrending)

def formula_1_detrending_deviation(x_0, degree=20):
    """
    Calcula la desviación delta_(n) entre la señal original x_0 
    y el ajuste polinomial de grado n.
    
    Parámetros:
        x_0 (array_like): Señal ECG original.
        degree (int): Grado del polinomio de ajuste (n = 20 según el artículo).
        
    Retorna:
        delta_n (float): Desviación (Error Cuadrático Medio).
        x_detrended (np.ndarray): Señal destendenciada (x_0 - x_n).
    """
    x_0 = np.asarray(x_0, dtype=float)
    k = len(x_0)
    t = np.arange(k)
    
    # Ajuste polinomial de grado n
    poly_coeffs = np.polyfit(t, x_0, degree)
    x_n = np.polyval(poly_coeffs, t)
    
    # Fórmula 1: Error cuadrático medio
    delta_n = np.sum((x_0 - x_n) ** 2) / k
    x_detrended = x_0 - x_n
    
    return delta_n, x_detrended

# FÓRMULA 2: Compresión (Normalización Min-Max)

def formula_2_compression(x):
    """
    Escala la señal x al rango [0, 1] mediante compresión min-max.
    
    Parámetros:
        x (array_like): Señal de entrada (e.g. señal destendenciada).
        
    Retorna:
        c (np.ndarray): Señal comprimida en el rango [0, 1].
    """
    x = np.asarray(x, dtype=float)
    x_min = np.min(x)
    x_max = np.max(x)
    
    if x_max == x_min:
        return np.zeros_like(x)
    
    c = (x - x_min) / (x_max - x_min)
    return c

# FÓRMULA 3: Biespectro

def formula_3_bispectrum(x, nperseg=8192):
    """
    Calcula el Biespectro b(f1, f2) promediado entre N segmentos.
    
    Parámetros:
        x (array_like): Señal unidimensional.
        nperseg (int): Longitud de cada segmento (8192 según el artículo).
        
    Retorna:
        b (np.ndarray 2D complex): Matriz triangular de Biespectro b(f1, f2).
    """
    x = np.asarray(x, dtype=float)
    k = len(x)
    N = k // nperseg
    
    if N < 1:
        N = 1
        nperseg = k

    # 1. Segmentación
    x_segmented = x[:N * nperseg].reshape(N, nperseg)
    
    # 2. Remoción de la media (DC) por segmento
    x_segmented = x_segmented - np.mean(x_segmented, axis=1, keepdims=True)
    
    # 3. Ventaneo Hanning para mitigar la fuga espectral (leakage)
    window = np.hanning(nperseg)
    x_segmented = x_segmented * window
    
    # 4. Transformada Rápida de Fourier (FFT) por segmento
    X = np.fft.fft(x_segmented, axis=1)
    
    num_freqs = nperseg // 2
    b = np.zeros((num_freqs, num_freqs), dtype=complex)
    
    # 5. Cálculo vectorizado del Biespectro promediado sobre los N segmentos
    for f1 in range(num_freqs):
        f2_max = num_freqs - f1
        # Promedio sobre los N segmentos: 1/N * sum( X_k(f1) * X_k(f2) * X_k*(f1+f2) )
        b[f1, :f2_max] = np.mean(
            X[:, f1:f1+1] * X[:, :f2_max] * np.conj(X[:, f1:f1+f2_max]), 
            axis=0
        )
            
    return b

# FÓRMULA 4: Bicoherencia

def formula_4_bicoherence(x, fs=500.0, nperseg=256, noverlap=None):
    """
    Calcula la Bicoherencia normalizada B(f1, f2) con traslape de segmentos
    y enmascaramiento de la región no válida (f1 + f2 > Nyquist).
    """
    if noverlap is None:
        noverlap = nperseg // 2  # Asigna automáticamente el 50% de traslape
        
    x = np.asarray(x, dtype=float)
    x = x - np.mean(x)  # Eliminar DC global
    
    step = nperseg - noverlap
    if step <= 0:
        raise ValueError("noverlap debe ser estrictamente menor que nperseg.")
        
    N = (len(x) - noverlap) // step  # Número total de segmentos
    
    if N < 5:
        raise ValueError(f"Señal demasiado corta ({len(x)} muestras) para nperseg={nperseg}. Reduce nperseg.")

    # 1. Segmentación con traslape (Welch-like)
    segments = np.array([x[i * step : i * step + nperseg] for i in range(N)])
    
    # Detrending y Ventaneo por segmento
    segments = segments - np.mean(segments, axis=1, keepdims=True)
    window = np.hanning(nperseg)
    segments = segments * window
    
    # 2. FFT por segmento
    X = np.fft.fft(segments, axis=1)
    
    num_freqs = nperseg // 2
    num = np.zeros((num_freqs, num_freqs), dtype=complex)
    den1 = np.zeros((num_freqs, num_freqs), dtype=float)
    den2 = np.zeros((num_freqs, num_freqs), dtype=float)
    
    # 3. Acumulación vectorial sobre los N segmentos
    for f1 in range(num_freqs):
        f2_max = num_freqs - f1  # Garantiza f1 + f2 <= Nyquist
        
        X_f1 = X[:, f1:f1+1]                         
        X_f2 = X[:, :f2_max]                         
        X_f12 = np.conj(X[:, f1:f1+f2_max])           
        
        # Numerador: Biespectro promedio
        num[f1, :f2_max] = np.mean(X_f1 * X_f2 * X_f12, axis=0)
        
        # Denominador: Potencias promediadas (Normalización Kim & Powers)
        den1[f1, :f2_max] = np.mean(np.abs(X_f1 * X_f2)**2, axis=0)
        den2[f1, :f2_max] = np.mean(np.abs(X_f12)**2, axis=0)
        
    # Bicoherencia al cuadrado
    den = den1 * den2
    den[den == 0] = np.nan
    bic = (np.abs(num)**2) / den
    
    # 4. Máscara triangular para ocultar valores fuera del dominio f1 + f2 > Nyquist
    mask = np.ones((num_freqs, num_freqs), dtype=bool)
    for i in range(num_freqs):
        mask[i, num_freqs - i:] = False
        
    bic[~mask] = np.nan
    
    return bic, N

Fórmula 3 - Biespectro (significado y utilidad):
   $$b(f_1, f_2) = \sum_{j=1}^{N} X_j(f_1) \, X_j(f_2) \, X_j^*(f_1 + f_2)$$
Calcula la transformada de Fourier de la correlación de tercer orden promediando sobre $N$ segmentos de señal, lo cual implica que cumple funciones cruciales en el análisis de señales complejas como el ECG que la transformada de Fourier convencional (espectro de potencia) no puede realizar [10]:

*Detección de interacciones no lineales (Acoplamiento Cuadrático de Fase): Identifica si dos componentes de frecuencia $f_1$ y $f_2$ interactúan no linealmente entre sí para generar una tercera frecuencia ($f_1 + f_2$). Esto ayuda a modelar la dinámica no lineal subyacente de la actividad eléctrica del corazón.

*Inmunidad al ruido Gaussiano: Estadísticamente, los momentos de tercer orden de cualquier ruido Gaussiano (sea blanco o coloreado) son teóricamente iguales a cero. El biespectro elimina o mitiga automáticamente el ruido Gaussiano aditivo en el ECG sin alterar las componentes útiles de la señal.

*Preservación de la información de fase: El espectro de potencia tradicional descarta la fase (al calcular el módulo al cuadrado de la transformada). El biespectro conserva la información de fase, lo cual es fundamental para preservar la morfología exacta de la señal cardíaca (las formas de las ondas P, QRS y T).

*Extracción de características para diagnóstico: Produce un mapa bidimensional $b(f_1, f_2)$ que actúa como una "huella digital" del estado del corazón. Las métricas extraídas de esta matriz se utilizan como entradas (features) en modelos de inteligencia artificial para clasificar o detectar patologías y arritmias cardíacas.  


A continuación se muestra un ejemplo para ilustrar la aplicación de las 4 funciones creadas en el presente literal:

## 4.Frecuencia de máxima potencia de las señales

4. Para cada señal extraer la frecuencia que contiene la máxima potencia 
(15%). 

## 5.Suma de frecuencias significativas de cada señal (sFSB)

5. Para cada señal, sumar el número de frecuencias que son significativas para bicoherencia (aquellas con un valor superior a $\sqrt{\dfrac{9.2}{2 \cdot N}}$ (10 %)).

## 6. Rutina aplicada al Dataframe para el registro, tipo de patología, la fMP y frecuencias significativas

6. Crear una rutina que aplique sobre todos los archivos de la base de datos las rutinas 3 al 5 y almacene los resultados en un dataframe donde se pueda registrar el registro, tipo de patología y la frecuencia de máxima potencia (fMP) y suma de frecuencias significativas en bicoherencia (sFSB) (10 %)

<div align="center">

| Registro | Estado | fMP | sFSB |
|:--------:|:------:|:---:|:----:|
|          |        |     |      |
|          |        |     |      |

</div>

In [44]:
RUTA_RESULTADOS = "resultados_punto6.csv"      # CSV donde se guarda y se retoma el progreso
df_diag = pd.read_excel(RUTA_DIAGNOSTICOS)     # tabla de diagnósticos completa

if os.path.exists(RUTA_RESULTADOS):
    os.remove(RUTA_RESULTADOS)

In [45]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch

# Valores que la función necesita antes de definirse (los valores por defecto se evalúan al definir la función)
RUTA_RESULTADOS = "resultados_punto6.csv"      # CSV donde se guarda y se retoma el progreso
if "df_diag" not in globals():
    df_diag = pd.read_excel(RUTA_DIAGNOSTICOS)  # tabla de diagnósticos completa


def procesar_base_completa(df_diag, carpeta=CARPETA_ECG, fs=FS, ruta_salida=RUTA_RESULTADOS, guardar_cada=500, nperseg_p3=256, f_max_plot=50.0, graficar=True):
    """
    GRAN FUNCIÓN UNIFICADA:
    Encapsula el 100% de las operaciones, fórmulas, graficación y procesamiento 
    de los Puntos 1 al 5/6 sobre la base de datos ECG sin requerir funciones externas.
    Con graficar=False omite las figuras por ritmo y procesa directamente la base.
    """
    
    # 1. FUNCIONES INTERNAS (Encapsuladas internamente)
    
    def _cargar_derivacion_II(nombre_archivo):
        """Lee el CSV de un registro y devuelve la derivación II."""
        if not str(nombre_archivo).endswith('.csv'):
            nombre_archivo = f"{nombre_archivo}.csv"
        ruta = os.path.join(carpeta, nombre_archivo)
        ecg = pd.read_csv(ruta)
        return (ecg['II'] if 'II' in ecg.columns else ecg.iloc[:, 1]).to_numpy(dtype=float)

    def _formula_1_detrending(x_0, degree=5):
        """Fórmula 1: Ajuste polinomial y eliminación de tendencia."""
        x_0 = np.asarray(x_0, dtype=float)
        t = np.arange(len(x_0))
        coeffs = np.polyfit(t, x_0, degree)
        x_n = np.polyval(coeffs, t)
        delta_n = np.mean((x_0 - x_n) ** 2)
        return delta_n, x_0 - x_n

    def _formula_2_compression(x):
        """Fórmula 2: Compresión Min-Max en el rango [0, 1]."""
        x = np.asarray(x, dtype=float)
        x_min, x_max = x.min(), x.max()
        if x_max == x_min:
            return np.zeros_like(x)
        return (x - x_min) / (x_max - x_min)

    def _formula_3_bispectrum(x, nperseg=256):
        """Fórmula 3: Cálculo del Biespectro."""
        x = np.asarray(x, dtype=float)
        nperseg = min(nperseg, len(x))
        if 'formula_3_bispectrum' in globals():
            return globals()['formula_3_bispectrum'](x, nperseg=nperseg)
        # Implementación de respaldo directa por TFD por segmentos
        step = nperseg // 2
        n_segments = (len(x) - nperseg) // step + 1
        if n_segments < 1:
            n_segments = 1
            nperseg = len(x)
        window = np.hanning(nperseg)
        fft_len = nperseg // 2 + 1
        biesp = np.zeros((fft_len, fft_len), dtype=complex)
        for i in range(n_segments):
            seg = x[i*step : i*step + nperseg]
            if len(seg) < nperseg: continue
            X = np.fft.rfft(seg * window)
            for f1 in range(fft_len):
                for f2 in range(fft_len - f1):
                    biesp[f1, f2] += X[f1] * X[f2] * np.conj(X[f1 + f2])
        return biesp / max(n_segments, 1)

    def _formula_4_bicoherence(x, nperseg=256):
        """Fórmula 4: Bicoherencia al cuadrado."""
        if 'formula_4_bicoherence' in globals():
            return globals()['formula_4_bicoherence'](x, fs=fs, nperseg=nperseg)
        biesp = _formula_3_bispectrum(x, nperseg=nperseg)
        num = np.abs(biesp) ** 2
        denom = np.nanmax(num) if np.nanmax(num) != 0 else 1.0
        return np.clip(num / denom, 0, 1), 1

    def _frecuencia_maxima_potencia(signal):
        """Punto 4: Frecuencia de Máxima Potencia (fMP) vía Welch PSD."""
        _, x_det = _formula_1_detrending(signal, degree=5)
        x = _formula_2_compression(x_det)
        x = x - np.mean(x)
        nperseg = min(2048, len(x))
        f, pxx = welch(x, fs=fs, window='hann', nperseg=nperseg, noverlap=nperseg // 2)
        banda = (f >= 0.5) & (f <= 50.0)
        i_max = np.argmax(pxx[banda])
        return f[banda][i_max], pxx[banda][i_max], f, pxx

    def _suma_frecuencias_significativas(signal):
        """Punto 5: Suma de Frecuencias Significativas del Biespectro (sFSB)."""
        if 'suma_frecuencias_significativas' in globals():
            # Tu función de la celda 72 devuelve (sFSB, N, umbral); aquí solo se necesita el conteo
            return globals()['suma_frecuencias_significativas'](signal, fs=fs)
        _, x_det = _formula_1_detrending(signal, degree=5)
        x = _formula_2_compression(x_det)
        bic, _ = _formula_4_bicoherence(x, nperseg=nperseg_p3)
        return float(np.nansum(bic)), bic, None, None

    def _graficar_puntos_1_a_3(senal, ritmo, registro):
        """Genera el panel 2x2 con las gráficas de las Fórmulas 1 a 4."""
        delta_n, x_det = _formula_1_detrending(senal, degree=5)
        x_comp = _formula_2_compression(x_det)
        biesp = _formula_3_bispectrum(x_comp, nperseg=nperseg_p3)
        bic2, N = _formula_4_bicoherence(x_comp, nperseg=nperseg_p3)

        t = np.arange(len(senal)) / fs
        n_f = biesp.shape[0]
        freqs = np.linspace(0, fs / 2, n_f)
        idx = np.searchsorted(freqs, f_max_plot)
        ext = [freqs[0], freqs[idx - 1], freqs[0], freqs[idx - 1]]
        nombre = NOMBRES_RITMO.get(ritmo, ritmo) if 'NOMBRES_RITMO' in globals() else ritmo

        fig, axes = plt.subplots(2, 2, figsize=(12, 9))

        # 1. Destendenciamiento
        axes[0, 0].plot(t, senal, color='gray', alpha=0.7, label='Original')
        axes[0, 0].plot(t, x_det, color='blue', label='Destendenciada (F1)')
        axes[0, 0].set_title('1. Ajuste y eliminación de tendencia'); axes[0, 0].grid(True)
        axes[0, 0].set_xlabel('Tiempo (s)'); axes[0, 0].set_ylabel('Amplitud'); axes[0, 0].legend()

        # 2. Compresión
        axes[0, 1].plot(t, x_comp, color='green')
        axes[0, 1].set_title('2. Señal comprimida [0, 1]'); axes[0, 1].grid(True)
        axes[0, 1].set_xlabel('Tiempo (s)'); axes[0, 1].set_ylabel('Amplitud normalizada')

        # 3. Biespectro Logarítmico
        im3 = axes[1, 0].imshow(np.log1p(np.abs(biesp[:idx, :idx])), cmap='viridis', origin='lower', extent=ext, aspect='equal')
        axes[1, 0].set_title(r'3. Biespectro logarítmico $\log(1 + |b|)$', fontweight='bold')
        axes[1, 0].set_xlabel('Frecuencia $f_2$ (Hz)'); axes[1, 0].set_ylabel('Frecuencia $f_1$ (Hz)')
        fig.colorbar(im3, ax=axes[1, 0], fraction=0.046, pad=0.04)

        # 4. Bicoherencia
        im4 = axes[1, 1].imshow(bic2[:idx, :idx], cmap='inferno', origin='lower', vmin=0, vmax=1, extent=ext, aspect='equal')
        axes[1, 1].set_title(r'4. Bicoherencia $B^2(f_1, f_2)$', fontweight='bold')
        axes[1, 1].set_xlabel('Frecuencia $f_2$ (Hz)'); axes[1, 1].set_ylabel('Frecuencia $f_1$ (Hz)')
        fig.colorbar(im4, ax=axes[1, 1], fraction=0.046, pad=0.04)

        plt.suptitle(f"Puntos 1-3 | {ritmo} – {nombre} | {registro}", fontweight='bold')
        plt.tight_layout(); plt.show()

    # 2. ETAPA A: GRÁFICAS Y MUESTRAS REPRESENTATIVAS POR RITMO (Puntos 1 a 3)
    if graficar:
        print("--- 1. Generando gráficas del Punto 3 por cada tipo de arritmia ---")
        tipos_arritmia = list(df_diag['Rhythm'].value_counts().index)
        
        for ritmo in tipos_arritmia:
            try:
                registro = str(df_diag.loc[df_diag['Rhythm'] == ritmo, 'FileName'].iloc[0])
                senal = _cargar_derivacion_II(registro)
                _graficar_puntos_1_a_3(senal, ritmo, registro)
            except Exception as e:
                print(f"No se pudo graficar {ritmo}: {e}")

    # 3. ETAPA B: PROCESAMIENTO DE LA BASE DE DATOS COMPLETA (Puntos 4 y 5)
    print("\n--- 2. Procesando la base de datos completa (fMP y sFSB) ---")
    resultados, hechos = [], set()
    if os.path.exists(ruta_salida):
        try:
            previo = pd.read_csv(ruta_salida)
            resultados = previo.to_dict('records')
            hechos = set(previo['Registro'].astype(str))
            print(f"Se retoma el progreso guardado: {len(hechos)} registros ya procesados.")
        except (pd.errors.EmptyDataError, KeyError):
            # CSV vacío o dañado de una corrida fallida: se empieza desde cero
            print(f"'{ruta_salida}' está vacío o dañado; se procesa desde cero.")
        
    errores = []
    pendientes = df_diag[~df_diag['FileName'].astype(str).isin(hechos)]
    
    for i, (registro, estado) in enumerate(zip(pendientes['FileName'].astype(str), pendientes['Rhythm']), 1):
        try:
            senal = _cargar_derivacion_II(registro)
            fmp = _frecuencia_maxima_potencia(senal)[0]
            sfsb = _suma_frecuencias_significativas(senal)[0]   # el primer valor es el conteo en ambas versiones
            resultados.append({'Registro': registro, 'Estado': estado, 'fMP': fmp, 'sFSB': sfsb})
        except Exception as e:
            errores.append((registro, str(e)))
            
        if i % guardar_cada == 0:
            pd.DataFrame(resultados).to_csv(ruta_salida, index=False)
            print(f"Progreso: {i}/{len(pendientes)} registros procesados... (errores: {len(errores)})")

    df_res = pd.DataFrame(resultados)
    df_res.to_csv(ruta_salida, index=False)
    print(f"Listo: {len(df_res)} registros procesados exitosamente, {len(errores)} errores.")
    if errores:
        print("Primeros errores:", errores[:3])
    
    return df_res, errores


In [46]:
df_resultados, errores = procesar_base_completa(df_diag)
display(df_resultados.head())

--- 1. Generando gráficas del Punto 3 por cada tipo de arritmia ---
No se pudo graficar SB: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BIOSEÑALES\\PROYECTOS\\ECG\\ECGData\\MUSE_20180112_073319_29000.csv'
No se pudo graficar SR: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BIOSEÑALES\\PROYECTOS\\ECG\\ECGData\\MUSE_20180209_172046_21000.csv'
No se pudo graficar AFIB: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BIOSEÑALES\\PROYECTOS\\ECG\\ECGData\\MUSE_20180113_171327_27000.csv'
No se pudo graficar ST: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BIOSEÑALES\\PROYECTOS\\ECG\\ECGData\\MUSE_20180113_135023_96000.csv'
No se pudo graficar SVT: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BIOSEÑALES\\PROYECTOS\\ECG\\ECGData\\MUSE_20180111_155633_99000.csv'
No se pudo graficar AF: [Errno 2] No such file or directory: 'C:\\Users\\12345\\Downloads\\UDEA\\BI

""


## 7. Estadística descriptiva para cada arritmia

7. Para las señales de análisis del punto 2 aplicar las rutinas del punto 3 al 
5  y  hacer  un  informe  que  permita  evidenciar  las  diferencias  entre  las 
señales las diferentes arritmias utilizando las herramientas de estadística 
descriptiva (5%). 

In [47]:
if not df_resultados.empty:
    display(
        df_resultados.groupby("Estado")[["fMP", "sFSB"]]
        .agg(["count", "mean", "median", "std"])
        .round(2)
    )
else:
    print("No se procesaron registros correctamente. Revisa `errores`.")

No se procesaron registros correctamente. Revisa `errores`.


### 7. Diferencias entre arritmias (estadística descriptiva)

Se compararon los 8 ritmos con n ≥ 121 (n = 10.615) usando mediana e IQR, porque ni fMP ni sFSB son normales (Shapiro-Wilk, p < 0,01 en todos los ritmos) y hay valores atípicos.

**Frecuencia de máxima potencia (fMP).**

| Ritmo | Mediana (Hz) | IQR (Hz) |
|---|---|---|
| SB | 2,93 | 2,69 |
| SA | 3,42 | 3,17 |
| AFIB | 3,42 | 2,20 |
| SR | 3,91 | 1,71 |
| AF (flutter) | 3,91 | 2,44 |
| AT | 4,39 | 1,71 |
| SVT | 4,88 | 2,69 |
| ST | 5,37 | 2,93 |

- La fMP ordena los ritmos sobre todo por la frecuencia cardíaca y no por la patología. SB (el ritmo más lento) tiene la mediana más baja y ST (el más rápido) la más alta, casi el doble. SVT y AT, también taquicardias, quedan por encima de SR.
- AFIB y flutter **no** desplazan la fMP hacia frecuencias altas: AFIB (3,42 Hz) queda por debajo de SR (3,91 Hz). Lo que distingue a AFIB es su dispersión (DE = 2,94 Hz, con atípicos hasta ≈ 50 Hz), consistente con un ritmo irregular que reparte la potencia en varias componentes.
- Las medianas (3–5 Hz) son mucho mayores que la frecuencia del pulso (≈ 1 Hz). El pico dominante corresponde a un armónico o a la energía de QRS y T, por lo que la fMP es un indicador indirecto del ritmo.

**Suma de frecuencias significativas en bicoherencia (sFSB).**

| Grupo | Ritmos | Mediana sFSB | % de pares significativos |
|---|---|---|---|
| Alto | ST, SR, SA, SB | 25.656 – 28.291 | 78 – 83 % |
| Bajo | AFIB, SVT, AF, AT | 20.313 – 22.888 | 64 – 71 % |

- Los ritmos sinusales (ST, SR, SA, SB) presentan más acoplamiento cuadrático de fase: sus latidos son regulares y repiten la misma morfología, de modo que los armónicos de la señal están acoplados en fase.
- AFIB tiene el sFSB más bajo (mediana 20.313; ≈ 64 % de los pares). Es coherente con la pérdida de regularidad y de actividad auricular organizada. SVT, flutter y AT también quedan por debajo de los ritmos sinusales.
- sFSB discrimina mejor que fMP entre sinusal y no sinusal. La diferencia SR–AFIB tiene δ de Cliff = 0,59 (grande), mientras que SR–AFIB en fMP solo alcanza 0,13 (despreciable). En cambio, sFSB no separa ST de SR (p Holm no significativo en la tabla de pares), y fMP sí los separa.
- El umbral 9,2/(2N) = 0,256 lo superan entre 64 % y 83 % de los pares, así que el sFSB es sensible al contenido armónico del complejo QRS. Se interpreta de forma comparativa entre ritmos y no como un conteo de interacciones fisiológicas aisladas.

**Síntesis.** fMP separa los ritmos por velocidad (SB < SR < ST) y sFSB los separa por regularidad (sinusales > AFIB, SVT, flutter y AT). Usadas juntas son complementarias: ST y SR se diferencian por fMP, y SR y AFIB por sFSB. Las diferencias son estadísticamente significativas (punto 8) y su magnitud se evalúa con ε² y δ de Cliff, no solo con el p-valor.

## 8. Diferencia estadística a través de pruebas estadísticas

8. Indicar  si  hay  o  no  diferencias  estadísticas  entre  las  características 
espectrales del punto 4 y 5 para las poblaciones de estudio (5%)

**Hipótesis formuladas** (α = 0,05; aplican a fMP y a sFSB)

| Prueba | H₀ | H₁ |
|---|---|---|
| Shapiro-Wilk (por ritmo) | La variable sigue una distribución normal | La variable no sigue una distribución normal |
| Levene (Brown-Forsythe) | $\sigma_1^2 = \sigma_2^2 = \dots = \sigma_8^2$ | Al menos una varianza difiere |
| Kruskal-Wallis | La variable tiene la misma distribución en los 8 ritmos | Al menos un ritmo tiene una distribución distinta |
| Mann-Whitney (por pares, Holm) | $P(X_A > X_B) = 0{,}5$ | $P(X_A > X_B) \neq 0{,}5$ |

## 9. Análisis de resultados y conclusiones

9. Hacer un informe con todos los puntos anteriores. El informe debe 
incluir análisis de resultados y conclusiones (15%). Trabajo sin sustentar 
no se califica

### 9.1 Resumen del trabajo

Se analizaron 10.646 electrocardiogramas de 12 derivaciones de la base ChapmanECG [4] (10 s, 500 Hz, derivación II). El trabajo siguió cinco pasos:
1. Se revisaron tres artículos de clasificación de arritmias [1]–[3].
2. Se caracterizó la base de datos (prevalencia, sexo, edad) y se definió un subconjunto de entrenamiento.
3. Se programaron las fórmulas del artículo de Kotriwar et al. [10]: eliminación de tendencia, compresión, biespectro y bicoherencia.
4. Se extrajeron dos características espectrales por señal: la frecuencia de máxima potencia (fMP) y la suma de frecuencias significativas en bicoherencia (sFSB).
5. Se compararon ambas características entre ritmos con estadística descriptiva y pruebas de hipótesis.

### 9.2 Hallazgos de la caracterización de la base (punto 2)

La asociación entre sexo y ritmo es significativa pero pequeña (χ² = 242,8; V de Cramér = 0,151). Tras la corrección de Bonferroni, la bradicardia sinusal predomina en hombres, y el ritmo sinusal, la taquicardia sinusal y la supraventricular predominan en mujeres. Esto concuerda con la mayor frecuencia cardíaca basal y la mayor susceptibilidad a reentrada nodal descritas en mujeres [5], [6]. La edad tiene un peso mayor (V = 0,231): la fibrilación auricular pasa de casi inexistente en jóvenes a cerca del 45 % en mayores de 80 años, en línea con la degeneración del sistema de conducción [9], y la arritmia sinusal es característica de edades jóvenes [8].

Con estos hallazgos se definió un subconjunto de adultos (≥ 18 años; 10.381 registros) en cinco grupos: SB, SR, AFIB, ST y GSVT. Se excluyeron los menores de 18 años porque sus frecuencias cardíacas normales son más altas y generan falsos diagnósticos de taquicardia con umbrales de adulto [8]. También se agruparon las clases con 16 o menos casos, que no permiten un aprendizaje estable [11]. Como control de calidad, la suma de hombres de la Tabla 6 de Zheng et al. [2] excede el total del texto en 972 casos, y intercambiar hombres y mujeres en cuatro ritmos hace cuadrar las cifras, lo que apunta a una errata de la publicación.

Se implementaron las fórmulas de Kotriwar et al. [10]: ajuste y eliminación de tendencia, compresión, biespectro y bicoherencia. El biespectro conserva la fase y suprime el ruido gaussiano, y permite detectar acoplamiento cuadrático de fase entre componentes [12]. Se usó un polinomio de grado 5 en lugar de 20, porque en registros de 10 s un grado alto empieza a ajustar la morfología del propio ECG. La densidad espectral se estimó con el método de Welch [13] y la bicoherencia se normalizó según Kim y Powers [14], con segmentos de 512 muestras y 50 % de traslape (N = 18). El umbral de significancia fue 9,2/(2N) = 0,256, aplicado a la bicoherencia al cuadrado; es equivalente al criterio √(9,2/(2N)) sobre su raíz.



### 9.3 Resultados espectrales (puntos 3 a 6)

**Decisiones metodológicas.** Se usaron segmentos de 512 muestras (≈1 s) con 50 % de traslape, lo que da N = 18 segmentos por registro y un umbral de significancia de 9,2/(2N) = 0,256 aplicado sobre la bicoherencia al cuadrado. La fMP se obtuvo con el método de Welch (ventana de 2048 muestras, resolución de 0,244 Hz). El detrending se aplicó con grado 5; el artículo usa grado 20 sobre señales de 60 s, y un grado alto sobre 10 s empieza a ajustar la morfología del propio ECG.

Resultados por ritmo (8 ritmos, n = 10.615):

| Ritmo | n | fMP mediana (Hz) | sFSB mediana | sFSB media (% de pares significativos) |
|---|---|---|---|---|
| SB | 3.889 | 2,93 | 25.656 | 78,4 % |
| SR | 1.826 | 3,91 | 27.935 | 83,4 % |
| AFIB | 1.780 | 3,42 | 20.313 | 64,0 % |
| ST | 1.568 | 5,37 | 28.291 | 83,2 % |
| SVT | 587 | 4,88 | 22.888 | 65,7 % |
| AF (flutter) | 445 | 3,91 | 22.761 | 71,0 % |
| SA | 399 | 3,42 | 26.924 | 81,9 % |
| AT | 121 | 4,39 | 21.898 | 68,0 % |


La frecuencia de máxima potencia (fMP) ordena los ritmos principalmente por la frecuencia cardíaca. La mediana más baja corresponde a SB (2,93 Hz) y la más alta a ST (5,37 Hz), con SR (3,91 Hz) y las taquicardias SVT (4,88 Hz) y AT (4,39 Hz) en posiciones intermedias o altas. La fibrilación auricular no desplaza la fMP hacia arriba: su mediana (3,42 Hz) es menor que la de SR, y su rasgo distintivo es la dispersión (DE = 2,94 Hz). Además, las medianas de 3–5 Hz superan ampliamente la frecuencia del pulso (≈ 1 Hz), por lo que el pico dominante corresponde a un armónico o a la energía de los complejos QRS y T [10], y la fMP es un indicador indirecto del ritmo.

La suma de frecuencias significativas en bicoherencia (sFSB) separa los ritmos por su regularidad. Los ritmos sinusales (ST, SR, SA y SB) presentan medianas de 25.656 a 28.291 pares significativos (78–83 % del máximo de 32.385), mientras que AFIB, SVT, flutter y AT quedan entre 20.313 y 22.888 (64–71 %). Es coherente con que un ritmo regular repite la misma morfología y mantiene los armónicos acoplados en fase, mientras que la actividad auricular desorganizada rompe ese acoplamiento. Sin embargo, entre el 64 % y el 83 % de los pares supera el umbral, de modo que el sFSB debe interpretarse de forma comparativa entre ritmos y no como un conteo de interacciones fisiológicas individuales.

### 9.3 Pruebas de hipótesis

Ninguna de las dos variables sigue una distribución normal (Shapiro-Wilk, p < 0,01 en todos los ritmos), por lo que se usaron pruebas no paramétricas. Kruskal-Wallis [15] indicó diferencias entre ritmos y las comparaciones por pares se hicieron con Mann-Whitney y la corrección de Holm [16]. Con muestras de miles de registros casi cualquier diferencia resulta significativa, de modo que la magnitud se evaluó con la δ de Cliff [17]. Esta es grande entre SB y ST en fMP (δ = −0,65) y entre SR y AFIB en sFSB (δ = 0,59), pero despreciable entre SR y AFIB en fMP (δ = 0,13). Por eso las dos características son complementarias: fMP discrimina por velocidad y sFSB por regularidad.

El sexo y la edad tienen un efecto menor. Las diferencias por sexo son despreciables en ambas variables (|δ| ≤ 0,10) y, tras Holm, solo SB mantiene una diferencia significativa en fMP. En cambio, el sFSB disminuye con la edad (ρ de Spearman = −0,25) y se mantiene al controlar la frecuencia ventricular (ρ parcial = −0,24), mientras que fMP prácticamente no se relaciona con ella (ρ = 0,02). Esto sugiere que la edad actúa sobre la regularidad y no solo sobre la frecuencia cardíaca, y obliga a considerar la edad como posible confusor, porque los pacientes con AFIB son los de mayor edad (mediana de 74 años).

### 9.4 Limitaciones

La base es hospitalaria  está desbalanceada (3.889 casos de SB frente a 121 de AT), lo que condiciona cualquier clasificador [11]. El análisis usa solo la derivación II y 10 s por registro, mientras Zheng et al. [2] encontraron que usar las 12 derivaciones mejora el F1 en 1,4 %. Con N = 18 segmentos traslapados, la bicoherencia tiene poca resolución estadística y los segmentos no son independientes, lo que hace laxo el umbral. El grado del polinomio se redujo respecto del artículo [10] y la fMP se limitó a 0,5–50 Hz, decisiones que pueden afectar los valores. Finalmente, los registros mantienen otras condiciones cardíacas (columna *Beat*) cuyo efecto no se controló, y Zheng et al. [2] reportan que reducen el F1 entre 0,017 y 0,034.

### 9.5 Posibles mejoras

- Usar registros más largos o segmentos con menos traslape para aumentar N, o implementar la bicoherencia de la Ecuación 4 de Kotriwar et al. [10] y comparar ambas normalizaciones.
- Controlar la edad y las condiciones adicionales con un modelo multivariable (regresión o ANCOVA sobre rangos) en lugar de pruebas univariadas.
- Extraer características adicionales (variabilidad de la frecuencia cardíaca [18], intervalos RR, razones de potencia por banda) y usar las 12 derivaciones.
- Entrenar un clasificador sobre fMP y sFSB, con validación cruzada y manejo del desbalance (sobremuestreo o pérdida ponderada) [11], y compararlo con los resultados de [1] y [2].
- Reportar intervalos de confianza por bootstrap para las medianas y los tamaños de efecto.

### 9.6 Conclusiones

1. La base ChapmanECG es representativa de la de Zheng et al. [2] en orden de prevalencia y perfil demográfico, pero refleja una población hospitalaria, por lo que sirve para entrenar modelos y no para estimar prevalencias.
2. Las arritmias dependen más de la edad (V = 0,231) que del sexo (V = 0,151); la fibrilación auricular crece con la edad y la arritmia sinusal predomina en jóvenes.
3. Se seleccionó un subconjunto de adultos con cinco grupos (SB, SR, AFIB, ST y GSVT) para reducir el sesgo pediátrico y la escasez de clases raras.
4. Se programaron las cuatro fórmulas de [10] y se procesaron los 10.646 registros sin errores, obteniendo fMP y sFSB para cada uno.
5. fMP refleja principalmente la frecuencia cardíaca y sFSB la regularidad del ritmo; ambas difieren significativamente entre ritmos y son complementarias, aunque sFSB está condicionada por la edad.
6. Los resultados son descriptivos y sirven para orientar la selección de características; para un clasificador se requieren más variables, control de confusores y validación externa.
7. **La edad es el factor clínico que más se relaciona con el tipo de ritmo en esta base**, por encima del sexo. La fibrilación auricular es propia de personas de edad avanzada y la irregularidad sinusal, de personas jóvenes. Estas tendencias concuerdan con lo publicado en la literatura, lo que avala la calidad de la base.
8. **Las fórmulas de Kotriwar et al. no se pueden usar tal cual en registros de 10 s y requieren ajustes.** El grado del polinomio, la longitud de los segmentos y el cálculo del espectro modifican los resultados, por lo que cada decisión debió fundamentarse y comprobarse.
9. **Las características espectrales sí varían entre arritmias, principalmente porque cada arritmia presenta una frecuencia cardíaca diferente.** La fMP acompaña a los armónicos de la frecuencia cardíaca, y la sFSB depende de cómo esos armónicos coinciden con la resolución de la bicoherencia.




## 10. Bibliografía

[1] L.-R. Liu et al., “An arrhythmia classification approach via deep learning using single-lead ECG without QRS wave detection,” Heliyon, vol. 10, Art. no. e27200, Feb. 2024, doi: 10.1016/j.heliyon.2024.e27200.

[2] J. Zheng et al., “Optimal multi-stage arrhythmia classification approach,” Sci. Rep., vol. 10, Art. no. 2898, Feb. 2020, doi: 10.1038/s41598-020-59821-7.

[3] M. Gadaleta et al., “Prediction of atrial fibrillation from at-home single-lead ECG signals without arrhythmias,” npj Digit. Med., vol. 6, Art. no. 229, Dec. 2023, doi: 10.1038/s41746-023-00966-w.

[4] Zheng, Jianwei; Rakovski, Cyril; Danioko, Sidy; Zhang, Jianming; Yao, Hai; Hangyuan, Guo (2019). A 12-lead electrocardiogram database for arrhythmia research covering more than 10,000 patients. figshare. Collection. https://doi.org/10.6084/m9.figshare.c.4560497

[5] A. M. Russo et al., "Sex differences in arrhythmias," Nat. Rev. Cardiol., vol. 10, no. 4, pp. 201-209, 2013.

[6] L. A. MacFarlane et al., "Sex differences in electrophysiology and arrhythmias," Circ. Res., vol. 118, no. 6, pp. 1021-1031, 2016.

[7] J. Zheng et al., "A 12-lead electrocardiogram database for arrhythmia research covering more than 10,000 patients," Sci. Data, vol. 7, no. 1, p. 48, 2020.

[8] J. C. Perry y A. R. Garson, "Pediatric Arrhythmias," en Braunwald's Heart Disease: A Textbook of Cardiovascular Medicine, 11.ª ed., D. P. Zipes et al., Eds. Filadelfia, PA: Elsevier, 2019, pp. 883-911.

[9] L. Staerk et al., "Atrial Fibrillation: Epidemiology, Pathophysiology, and Clinical Outcomes," Circ. Res., vol. 120, no. 9, pp. 1501-1517, Abr. 2017.

[10] Y. Kotriwar, S. Kachhara, K. P. Harikrishnan, and G. Ambika, "Higher order spectral analysis of ECG signals," arXiv preprint arXiv:1809.08451, Sep. 2018.

[11] L. A. MacFarlane et al., “Sex differences in electrophysiology and arrhythmias,” *Circ. Res.*, vol. 118, no. 6, pp. 1021–1031, 2016.

[12] J. Zheng et al., “A 12-lead electrocardiogram database for arrhythmia research covering more than 10,000 patients,” *Sci. Data*, vol. 7, no. 1, art. no. 48, 2020.

[11] H. He and E. A. Garcia, “Learning from imbalanced data,” *IEEE Trans. Knowl. Data Eng.*, vol. 21, no. 9, pp. 1263–1284, Sep. 2009.

[12] C. L. Nikias and J. M. Mendel, “Signal processing with higher-order spectra,” *IEEE Signal Process. Mag.*, vol. 10, no. 3, pp. 10–37, Jul. 1993.

[13] P. D. Welch, “The use of fast Fourier transform for the estimation of power spectra: A method based on time averaging over short, modified periodograms,” *IEEE Trans. Audio Electroacoust.*, vol. AU-15, no. 2, pp. 70–73, Jun. 1967.

[14] Y. C. Kim and E. J. Powers, “Digital bispectral analysis and its applications to nonlinear wave interactions,” *IEEE Trans. Plasma Sci.*, vol. PS-7, no. 2, pp. 120–131, Jun. 1979.

[15] W. H. Kruskal and W. A. Wallis, “Use of ranks in one-criterion variance analysis,” *J. Amer. Statist. Assoc.*, vol. 47, no. 260, pp. 583–621, 1952.

[16] S. Holm, “A simple sequentially rejective multiple test procedure,” *Scand. J. Statist.*, vol. 6, no. 2, pp. 65–70, 1979.

[17] N. Cliff, “Dominance statistics: Ordinal analyses to answer ordinal questions,” *Psychol. Bull.*, vol. 114, no. 3, pp. 494–509, 1993.

[18] Task Force of the European Society of Cardiology and the North American Society of Pacing and Electrophysiology, “Heart rate variability: Standards of measurement, physiological interpretation, and clinical use,” *Circulation*, vol. 93, no. 5, pp. 1043–1065, 1996.